In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# 1. Initialize Spark
spark = SparkSession.builder \
    .appName("LendingClub_Gold_ML") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

# 2. Load the Parquet Data (This will be fast!)
print("Loading Silver data...")
df = spark.read.parquet("../data/silver_lending_club_cleaned.parquet")

# 3. Feature Engineering
df = df.withColumn("fico_band", 
    when(col("fico_range_low") < 650, "Subprime")
    .when((col("fico_range_low") >= 650) & (col("fico_range_low") < 700), "Near-Prime")
    .when((col("fico_range_low") >= 700) & (col("fico_range_low") < 750), "Prime")
    .otherwise("Super-Prime")
)

df = df.withColumn("dti_band",
    when(col("dti") < 10, "Low DTI")
    .when((col("dti") >= 10) & (col("dti") < 20), "Medium DTI")
    .when((col("dti") >= 20) & (col("dti") < 30), "High DTI")
    .otherwise("Very High DTI")
)

print(f"Data Loaded! Row count: {df.count()}")
df.select("loan_amnt", "int_rate", "fico_range_low", "fico_band", "dti", "dti_band", "is_default").show(5)

Loading Silver data...
Data Loaded! Row count: 1345349
+---------+--------+--------------+-----------+-----+----------+----------+
|loan_amnt|int_rate|fico_range_low|  fico_band|  dti|  dti_band|is_default|
+---------+--------+--------------+-----------+-----+----------+----------+
|  12000.0|   15.22|         670.0| Near-Prime|  3.8|   Low DTI|         0|
|   7125.0|   15.88|         695.0| Near-Prime|13.28|Medium DTI|         0|
|  24000.0|   14.33|         695.0| Near-Prime|12.52|Medium DTI|         0|
|  24000.0|    6.03|         755.0|Super-Prime| 7.08|   Low DTI|         0|
|  18000.0|   17.56|         695.0| Near-Prime|13.92|Medium DTI|         0|
+---------+--------+--------------+-----------+-----+----------+----------+
only showing top 5 rows



In [1]:
import setuptools
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# 1. Start Spark
spark = SparkSession.builder.appName("LendingClub_Gold_ML").config("spark.driver.memory", "2g").getOrCreate()

# 2. Load Silver Data
df = spark.read.parquet("../data/silver_lending_club_cleaned.parquet")

# 3. Feature Engineering
df = df.withColumn("fico_band", 
    when(col("fico_range_low") < 650, "Subprime")
    .when((col("fico_range_low") >= 650) & (col("fico_range_low") < 700), "Near-Prime")
    .when((col("fico_range_low") >= 700) & (col("fico_range_low") < 750), "Prime")
    .otherwise("Super-Prime")
)
df = df.withColumn("dti_band",
    when(col("dti") < 10, "Low DTI")
    .when((col("dti") >= 10) & (col("dti") < 20), "Medium DTI")
    .when((col("dti") >= 20) & (col("dti") < 30), "High DTI")
    .otherwise("Very High DTI")
)

# --- FIX: Force numeric columns to be actual numbers ---
numeric_cols = ["loan_amnt", "int_rate", "dti", "annual_inc", "installment", "fico_range_low", "fico_range_high"]
for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("float"))
# ------------------------------------------------------

# 4. Prepare Features
categorical_cols = ["fico_band", "dti_band", "home_ownership", "purpose", "term"]

indexers = [StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep") for c in categorical_cols]
assembler_inputs = [c + "_index" for c in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep")

pipeline = Pipeline(stages=indexers + [assembler])
df_ml = pipeline.fit(df).transform(df)

# 5. Split Train/Test
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Data successfully prepared for ML!")
print(f"Training rows: {train_data.count()}")
print(f"Testing rows: {test_data.count()}")

Data successfully prepared for ML!
Training rows: 1075996
Testing rows: 269353


In [2]:
# Convert a sample to Pandas (local driver memory)
train_pd = train_data.select("features", "is_default").sample(fraction=0.1, seed=42).toPandas()
test_pd = test_data.select("features", "is_default").sample(fraction=0.1, seed=42).toPandas()

# Extract the vector into an array for sklearn
import numpy as np
X_train = np.array([v.toArray() for v in train_pd["features"]])
y_train = train_pd["is_default"].values
X_test = np.array([v.toArray() for v in test_pd["features"]])
y_test = test_pd["is_default"].values



In [3]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

model = xgb.XGBClassifier(n_estimators=50, max_depth=5, learning_rate=0.1, use_label_encoder=False)
model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, preds)
print(f"XGBoost (local sample) AUC: {auc:.4f}")

d:\fabric-credit-risk-lending-club\venv\Lib\site-packages\xgboost\sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")


XGBoost (local sample) AUC: 0.7031


In [5]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

print("--- RUNNING VERIFICATION ---")

# 1. Get True Labels from the PySpark test set
y_true = np.array(test_data.select("is_default").collect()).ravel()

# 2. Get Features for prediction (using the same sample logic we used before to avoid memory crash)
# We extract a sample to match what we exported to Power BI
test_sample = test_data.select("features", "is_default").sample(fraction=0.1, seed=42).toPandas()

y_true_sample = test_sample["is_default"].values
X_test_sample = np.array([v.toArray() for v in test_sample["features"]])

# 3. Predict Probabilities
y_prob = model.predict_proba(X_test_sample)[:, 1]

# 4. Apply the 50% threshold (Step 2 of the checklist)
y_pred = (y_prob >= 0.5).astype(int)

# 5. Calculate Metrics
tn, fp, fn, tp = confusion_matrix(y_true_sample, y_pred, labels=[0, 1]).ravel()

print(f"Test rows (Sample): {len(y_true_sample):,}")
print(f"TN: {tn:,} | FP: {fp:,} | FN: {fn:,} | TP: {tp:,}")
print(f"Precision: {precision_score(y_true_sample, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true_sample, y_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_true_sample, y_pred, zero_division=0):.4f}")
print(f"AUC:       {roc_auc_score(y_true_sample, y_prob):.4f}")

--- RUNNING VERIFICATION ---


Test rows (Sample): 26,847
TN: 21,216 | FP: 238 | FN: 5,090 | TP: 303
Precision: 0.5601
Recall:    0.0562
F1-score:  0.1021
AUC:       0.7031


In [ ]:
import pandas as pd
import numpy as np

print("Exporting FULL test data (100%) for Power BI...")

# We removed the .sample() line to take the full dataset
test_sample = test_data.select(
    "addr_state", 
    "purpose", 
    "home_ownership", 
    "term",
    "loan_amnt", 
    "int_rate", 
    "fico_range_low", 
    "dti", 
    "annual_inc",          
    "emp_length",          
    "verification_status", 
    "issue_d",             
    "is_default", 
    "features"
).toPandas()  # <--- Notice the .sample() is gone

# 2. Extract features for prediction
X_test_full = np.array([v.toArray() for v in test_sample["features"]])

# 3. Predict probabilities
probabilities = model.predict_proba(X_test_full)[:, 1]

# 4. Add predictions back to the dataframe
test_sample["predicted_default_prob"] = probabilities

# 5. Drop the complex "features" vector column
test_sample = test_sample.drop(columns=["features"])

# 6. Export to CSV
csv_path = "../data/powerbi_credit_risk_predictions.csv"
test_sample.to_csv(csv_path, index=False)
print(f"Success! Data exported to: {csv_path}")
print(f"Total rows exported: {len(test_sample)}")



Exporting FULL test data (100%) for Power BI...
Success! Data exported to: ../data/powerbi_credit_risk_predictions.csv
Total rows exported: 269353


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Replace these with your actual test labels and predictions.
# y_test = ...
# y_pred = ...

tn, fp, fn, tp = confusion_matrix(
    y_test, y_pred, labels=[0, 1]
).ravel()

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))